In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KGramConstruction") \
    .getOrCreate()

sc = spark.sparkContext

26/03/04 14:45:05 WARN Utils: Your hostname, rajesh-pc resolves to a loopback address: 127.0.1.1; using 192.168.0.39 instead (on interface wlp1s0)
26/03/04 14:45:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 14:45:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/04 14:45:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/04 14:45:06 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/04 14:45:06 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
import random
import hashlib
from itertools import combinations

In [3]:
base_path = "/home/rajesh/CSL7100/Assignment2/minhash/"

files = [
    base_path + "D1.txt",
    base_path + "D2.txt",
    base_path + "D3.txt",
    base_path + "D4.txt"
]

In [4]:
docs = {}

for i, file in enumerate(files):
    with open(file, "r") as f:
        docs[f"D{i+1}"] = f.read().strip()


In [5]:
print("D1: " + docs["D1"][:50])
print("D2: " + docs["D2"][:50])
print("D3: " + docs["D3"][:50])
print("D4: " + docs["D4"][:50])


D1: apple ceo tim cook is spending some time in canada
D2: apple ceo tim cook is spending some time in canada
D3: as part of his one day tour of canada yesterday ti
D4: president trump who warned as a candidate about th


## BUILD ALL k-GRAMS ONCE (Question 1)

In [6]:
def char_kgrams(text, k):
    text = text.strip()        # remove leading/trailing spaces
    result = set()             # use set to remove duplicates

    for i in range(len(text) - k + 1):   # valid start positions
        gram = text[i:i+k]               # substring of length k
        result.add(gram)                 # add to set (no duplicates)

    return result

In [7]:
def word_2grams(text):
    words = text.strip().split()   # split into words
    result = set()                 # use set to remove duplicates

    for i in range(len(words) - 1):   # until second-last word
        gram = words[i] + " " + words[i+1]   # combine two words
        result.add(gram)                   # add to set

    return result

In [8]:
char2_sets = {}
char3_sets = {}
word2_sets = {}

for name, text in docs.items():

    # Character 2-grams (set removes duplicates)
    char2_sets[name] = char_kgrams(text,2)

    # Character 3-grams
    char3_sets[name] = char_kgrams(text,3)

    # Word 2-grams
    words = text.split()
    word2_sets[name] = word_2grams(text)

In [9]:
print(list(char2_sets["D1"])[:5])
print(list(char3_sets["D2"])[:5])
print(list(word2_sets["D2"])[:5])

['ry', 'ki', 'bl', 'pi', 'no']
['pas', 'ike', 't t', 'r m', 'azo']
['from the', 'cooks full', 'said developers', 'a hockey', 'visited the']


In [10]:
# Use already created 3-gram sets
S1 = char3_sets["D1"]
S2 = char3_sets["D2"]

In [11]:
random.seed(42)                 # fix randomness
p = 2147483647                  # large prime

#converts the a k gram string to a large interger value. use to map k-granm to number. 
def stable_hash(x):
    return int(hashlib.md5(x.encode()).hexdigest(), 16) #string→MD5 hash→hex string→big integer

In [41]:
import random
import time

m = 20011   # large prime > 10000

# Generate t hash functions of form (a*x + b) % m. Ths will return list of t -tuples (a,b)  for t differnt hash functions
def generate_hash_functions(t):
    funcs = []
    for _ in range(t):
        a = random.randint(1, m-1)   # random multiplier
        b = random.randint(0, m-1)   # random offset
        funcs.append((a, b))
    return funcs


# Compute MinHash signature for a set of shingles for the set of t different hash functions.
def compute_signature(shingle_set, hash_funcs):
    signature = []
    # for a element in shingles set find the minimun hash value for a particular has functions. this will done for t - hash functions.
    for a, b in hash_funcs:
        min_hash = float('inf')   # initialize minimum
        for sh in shingle_set:
            x = compute_signature(sh) # convert shingle to integer
            h = (a * x + b) % m   # apply hash function for the family of hash functions
            if h < min_hash:
                min_hash = h      # update minimum
        signature.append(min_hash)
    return signature


# Estimate similarity from signatures
def estimate_jaccard(sig1, sig2):
    matches = sum(1 for i in range(len(sig1)) if sig1[i] == sig2[i])  # count matches
    return matches / len(sig1)


# True Jaccard similarity
def true_jaccard(set1, set2):
    return len(set1 & set2) / len(set1 | set2)  # exact similarity

# Compute true Jaccard similarity between two sets
def true_jaccard(set1, set2):
    intersection = set1 & set2   # common elements
    union = set1 | set2          # all unique elements
    return len(intersection) / len(union)   # J(A,B) = |A∩B| / |A∪B|



In [52]:
# Use already created 3-gram sets
S1 = char3_sets["D1"]
S2 = char3_sets["D2"]


In [55]:
import time
import random
import numpy as np

run_max = 1
t_values = [20, 60, 150, 300, 600]

# Store results for each t
results = {t: {"similarities": [], "times": []} for t in t_values}

for count in range(run_max):

    for t in t_values:
    
        start_time = time.time()
    
        hash_functions = []
    
        for _ in range(t):
            a = random.randint(1, p-1)
            b = random.randint(0, p-1)
            hash_functions.append((a, b))
    
        sig1 = []
        sig2 = []
    
        for (a, b) in hash_functions:
    
            sig1.append(min((a * stable_hash(g) + b) % p for g in S1))
            sig2.append(min((a * stable_hash(g) + b) % p for g in S2))
    
        matches = sum(1 for i in range(t) if sig1[i] == sig2[i])
        estimated_similarity = matches / t
    
        end_time = time.time()
        elapsed_time = end_time - start_time
    
        # Store values
        results[t]["similarities"].append(estimated_similarity)
        results[t]["times"].append(elapsed_time)

        print(f"t = {t}, "
        f"Estimated Jaccard = {estimated_similarity:.4f}, "
        f"Time = {elapsed_time:.4f} seconds")

actual_similarity = true_jaccard(S1, S2)
print("Actual Jaccard Similarity (D1, D2):", actual_similarity)

t = 20, Estimated Jaccard = 1.0000, Time = 0.0307 seconds
t = 60, Estimated Jaccard = 0.9500, Time = 0.0787 seconds
t = 150, Estimated Jaccard = 0.9933, Time = 0.1987 seconds
t = 300, Estimated Jaccard = 0.9900, Time = 0.3973 seconds
t = 600, Estimated Jaccard = 0.9717, Time = 0.7850 seconds
Actual Jaccard Similarity (D1, D2): 0.977979274611399


In [54]:
import time
import random
import numpy as np

run_max = 10
t_values = [20, 60, 150, 300, 600]

# Store results for each t
results = {t: {"similarities": [], "times": []} for t in t_values}

for count in range(run_max):

    for t in t_values:
    
        start_time = time.time()
    
        hash_functions = []
    
        for _ in range(t):
            a = random.randint(1, p-1)
            b = random.randint(0, p-1)
            hash_functions.append((a, b))
    
        sig1 = []
        sig2 = []
    
        for (a, b) in hash_functions:
    
            sig1.append(min((a * stable_hash(g) + b) % p for g in S1))
            sig2.append(min((a * stable_hash(g) + b) % p for g in S2))
    
        matches = sum(1 for i in range(t) if sig1[i] == sig2[i])
        estimated_similarity = matches / t
    
        end_time = time.time()
        elapsed_time = end_time - start_time
    
        # Store values
        results[t]["similarities"].append(estimated_similarity)
        results[t]["times"].append(elapsed_time)

# Print aggregated statistics
print("\n===== Aggregated Results =====")

for t in t_values:
    mean_sim = np.mean(results[t]["similarities"])
    std_sim = np.std(results[t]["similarities"])
    
    mean_time = np.mean(results[t]["times"])
    std_time = np.std(results[t]["times"])
    
    print(f"\n--- t = {t} ---")
    print(f"Mean Estimated Jaccard : {mean_sim:.6f}")
    print(f"Std Dev Similarity     : {std_sim:.6f}")
    print(f"Mean Time (seconds)    : {mean_time:.6f}")
    print(f"Std Dev Time           : {std_time:.6f}")

print()
actual_similarity = true_jaccard(S1, S2)
print("Actual Jaccard Similarity (D1, D2):", actual_similarity)


===== Aggregated Results =====

--- t = 20 ---
Mean Estimated Jaccard : 0.985000
Std Dev Similarity     : 0.022913
Mean Time (seconds)    : 0.026801
Std Dev Time           : 0.000951

--- t = 60 ---
Mean Estimated Jaccard : 0.983333
Std Dev Similarity     : 0.012910
Mean Time (seconds)    : 0.079572
Std Dev Time           : 0.000946

--- t = 150 ---
Mean Estimated Jaccard : 0.976667
Std Dev Similarity     : 0.015563
Mean Time (seconds)    : 0.198376
Std Dev Time           : 0.001828

--- t = 300 ---
Mean Estimated Jaccard : 0.975000
Std Dev Similarity     : 0.009690
Mean Time (seconds)    : 0.394961
Std Dev Time           : 0.003139

--- t = 600 ---
Mean Estimated Jaccard : 0.976167
Std Dev Similarity     : 0.007712
Mean Time (seconds)    : 0.789640
Std Dev Time           : 0.003792

Actual Jaccard Similarity (D1, D2): 0.977979274611399


In [58]:
if spark:
    spark.stop()